In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [21]:
# Load IMDB word index
word_index=imdb.get_word_index()
#reverse_word_index=dict([(value,key) for (key,value) in word_index.items()])
reverse_word_index = {
    value: key for key, value in word_index.items()
}

In [22]:
#load model
model=load_model('simple_rnn_imdb.h5')
model.summary

<bound method Model.summary of <Sequential name=sequential_1, built=True>>

In [23]:
print(model.input_shape)

(None, 300)


In [24]:
#helper function
#1 function to decoded the review
def decoded_review(encoded_review):
  return ''.join( [
    reverse_word_index.get(i-3,'?') 
    for i in encoded_review if i >=4
    ])

In [28]:

def preprocess_text(text):
    words = text.lower().split()

    encoded_review = []

    for word in words:
        index = word_index.get(word)

        if index is None or index + 3 >= 10000:
            encoded_review.append(2)
        else:
            encoded_review.append(index + 3)

    padded_review = sequence.pad_sequences(
        [encoded_review],
        maxlen=300
    )

    return padded_review


In [29]:
test_input = preprocess_text(example_review)

print("Input shape:", test_input.shape)
print("Maximum word index:", test_input.max())
print("Minimum word index:", test_input.min())

Input shape: (1, 300)
Maximum word index: 5647
Minimum word index: 0


In [30]:
# prediction function
def predict_sentiment(text):
  preprocessed_input = preprocess_text(text)
  prediction = model.predict(preprocessed_input, verbose=0)[0][0]
  sentiment = "Positive" if prediction > 0.5 else "Negative"
  return sentiment, float(prediction)

  #[0][0]  used because model return the prediction in 2D array format in which first[0] select the first review(first row) and second[0] select the prediction score

In [31]:
# User input & prediction
example_review = "Avengers Endgame is one of the best superhero movies in the Marvel Cinematic Universe. The movie continues the story after the shocking events of Avengers Infinity War. The story is full of emotions, action, suspense and unexpected twists. All the actors have delivered excellent performances. Overall, it is an emotional and entertaining movie."

sentiment, score = predict_sentiment(example_review)

print(f"Sentiment: {sentiment}")
print(f"Score: {score:.4f}")


Sentiment: Positive
Score: 0.9505
